# Part 2: Simulation and experiments

Part 1 produced three hourly input datasets and merged them into one **state table** (`state_table.csv`).
Part 2 turns that table into a simulation:

1. Fix the **assumptions** (battery, generator, prices) in one place
2. Build an **engine** that applies physical limits and checks the energy balance every hour
3. Define the **strategies** that decide where energy comes from: `naive`, `reactive`, static-reserve `helios`, and a perfect-foresight `oracle`
4. Compare them on four grid-reliability scenarios
5. Test whether the conclusions survive **different random outage years** and **different system sizes**

The code lives in `src/simulation/` (engine, strategies, metrics, oracle, config). This notebook imports it, so the notebook and the scripts always agree.

In [ ]:
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# find the project root whether the notebook is run from the root or from notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "simulation").exists())
sys.path.insert(0, str(ROOT / "src"))

PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "data" / "results"
FIGURES_DIR = ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

from simulation import config as cfg
from simulation.engine import run_simulation
from simulation.metrics import avoided_vs, summarize_run
from simulation.oracle import make_oracle_strategy
from simulation.strategies import STRATEGIES

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("Project root:", ROOT)

## 9. Assumptions

Every number the simulation depends on lives in `src/simulation/config.py` and is tagged by how well it is supported:

- **sourced**: taken from a published figure
- **estimated**: an engineering estimate, to be verified against a spec sheet or quote
- **scenario**: a modelling choice we make, varied in sensitivity runs

The tariff is the Band A rate (at least 20 hours of supply a day) for the distribution company that serves Akwa Ibom; the fuel price is the national average petrol price reported by the NBS for July 2025. The battery cost and generator fuel curve are the least certain values and should be checked against a local quote and a spec sheet.

In [ ]:
assumptions = pd.DataFrame(
    [
        ("PV capacity (kW)", cfg.PV_CAPACITY_KW, "scenario (swept)"),
        ("PV system efficiency", cfg.SYSTEM_EFFICIENCY, "estimated"),
        ("Battery capacity (kWh)", cfg.BATTERY_CAPACITY_KWH, "scenario (swept)"),
        ("Battery usable depth of discharge", cfg.BATTERY_MAX_DOD, "estimated"),
        ("Battery round-trip efficiency", cfg.BATTERY_ROUND_TRIP_EFFICIENCY, "estimated"),
        ("Battery max charge / discharge (kW)", f"{cfg.BATTERY_MAX_CHARGE_KW} / {cfg.BATTERY_MAX_DISCHARGE_KW}", "estimated"),
        ("Battery initial state of charge", cfg.BATTERY_INITIAL_SOC, "scenario"),
        ("Battery cost (NGN per kWh of capacity)", cfg.BATTERY_COST_NGN_PER_KWH, "estimated (verify)"),
        ("Battery cycle life (cycles)", cfg.BATTERY_CYCLE_LIFE, "estimated"),
        ("Battery wear cost (NGN per kWh discharged)", round(cfg.BATTERY_WEAR_COST_PER_KWH, 1), "derived"),
        ("Generator rating (kW)", cfg.GENERATOR_RATED_KW, "estimated"),
        ("Generator fuel, no load (L/h)", cfg.GENERATOR_FUEL_L_PER_H_NO_LOAD, "estimated (verify)"),
        ("Generator fuel, full load (L/h)", cfg.GENERATOR_FUEL_L_PER_H_FULL_LOAD, "estimated (verify)"),
        ("Petrol price (NGN per litre)", cfg.FUEL_PRICE_NGN_PER_L, "sourced (NBS, July 2025)"),
        ("Grid tariff (NGN per kWh)", cfg.GRID_TARIFF_NGN_PER_KWH, "sourced (Band A)"),
        ("Generator maintenance", "not modelled", "omitted"),
    ],
    columns=["Assumption", "Value", "Type"],
)
assumptions

## 10. The state table and the engine

`state_table.csv` has one row per hour of the year: solar generation, household demand, and one grid-availability column per outage scenario (`grid_mild`, `grid_moderate`, `grid_severe`, `grid_extreme`).

**The engine knows physics, not strategy.** For every hour:

1. A strategy is asked what it *wants* (requested energy flows).
2. The engine clips those requests to what is physically possible: available solar, grid status, battery power and state-of-charge limits, generator rating.
3. The battery state of charge is updated (round-trip losses split between charging and discharging).
4. An **energy balance** is checked: energy in (solar, grid, generator) equals load served plus curtailed solar, net change in stored energy and battery losses. The run stops with an error if it is ever violated.

Load is served in a fixed order: **solar, then battery, then grid, then generator**. Anything left over is recorded as *unserved energy*; the engine never starts the generator on its own. Costs are not in the engine; they are computed afterwards from the recorded flows.

All four test files (engine physics, strategies, HELIOS rules and metrics, oracle) can be run from here:

In [ ]:
for name in ["test_engine", "test_strategies", "test_helios", "test_oracle"]:
    path = ROOT / "tests" / f"{name}.py"
    if not path.exists():
        print(f"{name:<16} (file not found)")
        continue
    out = subprocess.run([sys.executable, str(path)], capture_output=True, text=True, cwd=ROOT)
    lines = out.stdout.strip().splitlines() or out.stderr.strip().splitlines()
    print(f"{name:<16} {lines[-1] if lines else 'no output'}")

In [ ]:
state = pd.read_csv(PROCESSED_DIR / "state_table.csv", parse_dates=["timestamp"])
GRID_COLS = [c for c in state.columns if c.startswith("grid_")]

print("Rows:", len(state), "| first:", state["timestamp"].min(), "| last:", state["timestamp"].max())
print("Annual solar generation: ", round(state["solar_generation_kwh"].sum()), "kWh")
print("Annual household demand: ", round(state["demand_kwh"].sum()), "kWh")
print("Grid availability (%):   ", (100 * state[GRID_COLS].mean()).round(1).to_dict())
state.head()

## 11. The strategies

| Strategy | Idea |
|---|---|
| `naive` | A household with panels and no usable battery. Solar goes to the house, the grid supplies the rest, and the generator starts the moment the grid drops. The battery is never used. |
| `reactive` | A default inverter-battery household. Surplus solar charges the battery and the battery covers any shortfall whenever it can, grid up or not. The generator starts only when the grid is down and the battery cannot cover the load. No reserve, no anticipation. |
| `helios_r30/50/70` | `reactive` plus an **outage reserve**: while the grid is up, the battery may only be used down to a reserve level (30%, 50% or 70% state of charge). During an outage it may be used down to its floor. No forecasting. The reserve levels were fixed in advance from the demand profile (the morning peak needs roughly 1.5 kWh). |
| `oracle` | A **perfect-foresight benchmark**, not a machine-learning model. It reads the true future outage schedule from the table, keeps just enough energy for the coming outage, and tops up from the grid shortly before it. It shows the most that knowing outages in advance could be worth, and is a sensible foresight controller rather than a proven optimum. |

`reactive` is the main baseline. Comparing against `naive` alone would credit HELIOS for simply using the battery.

## 12. Results on one outage year

All strategies are run on all four grid scenarios, using the same solar, demand, hardware and prices. Only the outage timeline and the strategy change.

In [ ]:
runs, rows = {}, {}
for grid_col in GRID_COLS:
    scenario = grid_col.replace("grid_", "")
    strategies = {**STRATEGIES, "oracle": make_oracle_strategy(state, grid_col)}
    for name, strategy in strategies.items():
        res = run_simulation(state, strategy, grid_col=grid_col)
        runs[(scenario, name)] = res
        rows[(scenario, name)] = summarize_run(res)

summary = pd.DataFrame(rows).T.astype(float)
summary.index.names = ["scenario", "strategy"]
summary.to_csv(RESULTS_DIR / "notebook_summary.csv")

show = ["gen_hours", "fuel_l", "total_cost_ngn", "battery_cycles",
        "unserved_kwh", "solar_utilization_pct", "renewable_share_pct"]
summary[show].round(1)

In [ ]:
avoided_vs(summary, "reactive").round(0)

Positive numbers mean *better than `reactive`*. `unserved_kwh` is zero in every run, so the comparison is about cost and generator use, not blackouts.

In [ ]:
scenario_order = [c.replace("grid_", "") for c in GRID_COLS]
strategy_order = list(dict.fromkeys(summary.index.get_level_values("strategy")))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, metric, title, unit in [
    (axes[0], "total_cost_ngn", "Annual operating cost", "NGN"),
    (axes[1], "gen_hours", "Generator running hours", "hours"),
]:
    table = summary[metric].unstack("strategy").loc[scenario_order, strategy_order]
    table.plot.bar(ax=ax, width=0.85, rot=0)
    ax.set_title(title)
    ax.set_xlabel("grid scenario")
    ax.set_ylabel(unit)
    ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cost_and_generator_hours.png", bbox_inches="tight")
plt.show()

### What a bad day looks like

The day in the *extreme* scenario with the most generator running under `reactive`, shown for `reactive` and for the `oracle`. Stacked bars show where each hour's electricity came from; the line is the battery state of charge.

In [ ]:
def plot_day(ax, res, day, title):
    d = res[res["timestamp"].dt.date == day].reset_index(drop=True)
    parts = [("solar_to_load_kwh", "solar"), ("battery_to_load_kwh", "battery"),
             ("grid_to_load_kwh", "grid"), ("generator_to_load_kwh", "generator")]
    bottom = np.zeros(len(d))
    for col, label in parts:
        ax.bar(d["hour"], d[col], bottom=bottom, label=label, width=0.85)
        bottom += d[col].to_numpy()
    for h in d.loc[d["grid_available"] == 0, "hour"]:
        ax.axvspan(h - 0.5, h + 0.5, color="red", alpha=0.08, lw=0)
    ax.set_title(title)
    ax.set_xlabel("hour of day (red shading = grid outage)")
    ax.set_ylabel("kWh served")
    ax.set_xticks(range(0, 24, 2))
    ax2 = ax.twinx()
    ax2.plot(d["hour"], d["battery_soc"], color="black", marker=".", lw=1)
    ax2.set_ylim(0, 1.05)
    ax2.set_ylabel("battery state of charge")
    ax2.grid(False)

res_reactive = runs[("extreme", "reactive")]
worst_day = res_reactive.groupby(res_reactive["timestamp"].dt.date)["generator_running"].sum().idxmax()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
plot_day(axes[0], runs[("extreme", "reactive")], worst_day, f"reactive, {worst_day}")
plot_day(axes[1], runs[("extreme", "oracle")], worst_day, f"oracle, {worst_day}")
day_peak = res_reactive[res_reactive["timestamp"].dt.date == worst_day]["demand_kwh"].max()
axes[0].set_ylim(0, day_peak * 1.2)
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "worst_day.png", bbox_inches="tight")
plt.show()

## 13. Do the conclusions survive different outage years?

One random outage year can flatter or hurt a strategy. `src/run_sweep.py` regenerates the outage timeline for 30 different seeds (scenario parameters unchanged) and compares every strategy against `reactive` **seed by seed**. It runs for two assumed outage patterns:

- **uniform**: outages are equally likely to start at any hour (a control: there is nothing time-of-day-related to learn)
- **evening**: starts are three times more likely between 17:00 and 22:59 (an assumed peak-demand load-shedding pattern)

Neither pattern is measured data for Uyo; both are stated assumptions.

Columns: `saved_pct_mean` / `saved_pct_std` is the cost saving versus `reactive` (positive = cheaper); `seeds_beating_baseline` is the percentage of seeds in which the strategy was cheaper.

In [ ]:
def load_summary(name):
    path = RESULTS_DIR / name
    if not path.exists():
        print(f"{name} not found. Run:  python src/run_sweep.py" + (" --pattern evening" if "evening" in name else ""))
        return None
    return pd.read_csv(path)

seeds = {p: load_summary(f"seeds_{p}_summary.csv") for p in ["uniform", "evening"]}

for pattern, df in seeds.items():
    if df is not None:
        print(f"\n=== outage pattern: {pattern} ===")
        display(df[df["strategy"] != "naive"].drop(columns=["pv_kw", "battery_kwh", "unserved_kwh"]).round(1))

In [ ]:
available = {p: df for p, df in seeds.items() if df is not None}
if available:
    key_strategies = ["helios_r50", "helios_r70", "oracle"]
    fig, axes = plt.subplots(1, len(available), figsize=(7 * len(available), 4.5), sharey=True, squeeze=False)
    for ax, (pattern, df) in zip(axes[0], available.items()):
        for i, s in enumerate(key_strategies):
            sub = df[df["strategy"] == s].set_index("scenario").loc[scenario_order]
            x = np.arange(len(scenario_order)) + (i - 1) * 0.27
            ax.bar(x, sub["saved_pct_mean"], width=0.27, yerr=sub["saved_pct_std"], capsize=3, label=s)
        ax.axhline(0, color="black", lw=0.8)
        ax.set_xticks(range(len(scenario_order)))
        ax.set_xticklabels(scenario_order)
        ax.set_title(f"Cost saving vs reactive, 30 outage years ({pattern})")
        ax.set_ylabel("% cheaper than reactive (negative = costs more)")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "seeds_cost_saving.png", bbox_inches="tight")
    plt.show()

## 14. Does it still matter with a smaller system?

The base system (5 kW of panels, 5 kWh battery) is generous for a household using about 7 to 8 kWh a day, so even `reactive` copes well. This sweep shrinks the system: PV at 2, 3 and 5 kW and battery at 2.5 and 5 kWh, with fewer seeds (10) and a subset of strategies. The ranges were fixed before looking at results.

In [ ]:
sweeps = {p: load_summary(f"sweep_{p}_summary.csv") for p in ["uniform", "evening"]}
available = {p: df for p, df in sweeps.items() if df is not None}

for pattern, df in available.items():
    print(f"\n=== sizing sweep, outage pattern: {pattern}: cost saving vs reactive (%), extreme scenario ===")
    for strategy in ["helios_r50", "oracle"]:
        sub = df[(df["scenario"] == "extreme") & (df["strategy"] == strategy)]
        display(sub.pivot(index="pv_kw", columns="battery_kwh", values="saved_pct_mean").round(1).rename_axis(f"{strategy}: PV kW / battery kWh"))

In [ ]:
if available:
    fig, axes = plt.subplots(1, len(available), figsize=(6.5 * len(available), 4), squeeze=False)
    for ax, (pattern, df) in zip(axes[0], available.items()):
        sub = df[(df["scenario"] == "extreme") & (df["strategy"] == "reactive")]
        table = sub.pivot(index="pv_kw", columns="battery_kwh", values="gen_hours")
        im = ax.imshow(table.values, cmap="Reds", aspect="auto")
        ax.set_xticks(range(len(table.columns)), table.columns)
        ax.set_yticks(range(len(table.index)), table.index)
        for (i, j), v in np.ndenumerate(table.values):
            ax.text(j, i, f"{v:.0f}", ha="center", va="center")
        ax.set_xlabel("battery (kWh)")
        ax.set_ylabel("PV (kW)")
        ax.set_title(f"reactive generator hours, extreme ({pattern})")
        ax.grid(False)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "sizing_sweep_generator_hours.png", bbox_inches="tight")
    plt.show()

## 15. Where things stand

**Built and tested**
- Merged state table, assumptions file, physics engine with a per-hour energy-balance check
- Five strategy families, cost and renewable metrics, and a perfect-foresight benchmark
- Multi-seed and system-size robustness runs

**What the results so far say** (re-read the tables above after any rerun)
- Using the battery at all (`reactive` versus `naive`) is the largest single saving.
- A **static reserve** cuts generator hours but raises cost, because every kWh held back is bought from the grid at a higher price than the wear cost of stored solar. It only approaches break-even on the least reliable grids.
- The **oracle** shows that knowing *when* outages will happen is worth real money on unreliable grids and almost nothing on reliable ones. A trained predictor can only recover part of that.

**Limitations to state in the report**
- Demand and outages are synthetic; solar is real NASA POWER data.
- The two outage patterns are assumptions, not measured Uyo outage records.
- Several inputs are estimates (battery cost, generator fuel curve); results should be read as relative comparisons between strategies, not absolute bills.
- The base PV and battery are generous, which limits what any strategy can gain; the sizing sweep addresses this.
- Battery wear is modelled by throughput only (no calendar or temperature ageing); the generator has no maintenance cost.
- The oracle is a heuristic foresight controller, not a proven optimum.

**Next**
- Outage-risk predictor (logistic regression, judged on precision and recall against a "current state continues" baseline) and a predictive HELIOS using its output
- Final comparison: `reactive`, static reserve, predictive HELIOS, `oracle`